In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

## 2 lens plane (2p)

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = [0.4, 0.5]
zs = 1.5

# Angular diameter distances to lenses
Dl1 = Cosmology.angular_diameter_distance(cosmo, 0., zl[1])
Dl2 = Cosmology.angular_diameter_distance(cosmo, 0., zl[2])

# Distance ratios
Dls1 = Cosmology.angular_diameter_distance(cosmo, zl[1], zs)
Dls2 = Cosmology.angular_diameter_distance(cosmo, zl[2], zs)

# Angular diameter distance to source
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs);

# 2 lens plane + 2 point lens (2p-2p) case

In [ ]:
# Create image plane grid
x, y = Lenses.get_meshgrid(10, 10, 0.03);

multi_lens = [(z_d=zl[1], lens=:PointLens, D_d=Dl1, x_c= -1.0, y_c= 0, mass=5E11*MASS_SUN),
              (z_d=zl[2], lens=:PointLens, D_d=Dl2, x_c= +1.0, y_c= 0, mass=5E11*MASS_SUN)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

theta_E1 = Lenses.PointLens.einstein_angle(D_d=Dl1, D_ds=Dls1, D_s=Dos, mass=5E11*MASS_SUN)
theta_E2 = Lenses.PointLens.einstein_angle(D_d=Dl2, D_ds=Dls2, D_s=Dos, mass=5E11*MASS_SUN)
println(theta_E1," ", theta_E2)

src = Sources.gaussian(x, y, 0.1, 0.1, (0.0, 1.0), A=1.0)
fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

multi_lens = [(lens=:PointLens, z_d=zl[1], D_d=Dl1, x_c= -0.7, y_c= 0, mass=5E11*MASS_SUN),
              (lens=:PointLens, z_d=zl[2], D_d=Dl2, x_c= +0.7, y_c= 0, mass=5E11*MASS_SUN)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

multi_lens = [(lens=:PointLens, z_d=zl[1], D_d=Dl1, x_c= -0.4, y_c= 0, mass=5E11*MASS_SUN),
              (lens=:PointLens, z_d=zl[2], D_d=Dl2, x_c= +0.4, y_c= 0, mass=5E11*MASS_SUN)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

# 2 lens plane + 2 SIS lens

In [ ]:
# Create image plane grid
x, y = Lenses.get_meshgrid(6, 6, 0.03);

multi_lens = [(z_d=zl[1], lens=:SISLens, x_c= -1.0, y_c= -1.0, v_d=250E3),
              (z_d=zl[2], lens=:SISLens, x_c= +1.0, y_c= +1.0, v_d=250E3)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

src = Sources.gaussian(x, y, 0.1, 0.1, (0.0, 1.0), A=1.0)
fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

multi_lens = [(z_d=zl[1], lens=:SISLens, x_c= -0.7, y_c= -0.7, v_d=250E3),
              (z_d=zl[2], lens=:SISLens, x_c= +0.7, y_c= +0.7, v_d=250E3)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

multi_lens = [(z_d=zl[1], lens=:SISLens, x_c= -0.5, y_c= -0.5, v_d=250E3),
              (z_d=zl[2], lens=:SISLens, x_c= +0.5, y_c= +0.5, v_d=250E3)]
lens = Lenses.init_MultiPlaneLens(multi_lens)

fig, ax = MultiPlane.plot_image_plane(cosmo, lens, x, y, zs)
display(fig)

# Calculate quantities and plot

In [ ]:
# Get Jacobian
dxx, dyy, dxy, dyx = MultiPlane.get_jacobian(cosmo, lens, x, y, zs)

# Get convergence and shear
kappa = @. 0.5 * (dxx + dyy)
gamma1 = @. 0.5 * (dxx - dyy)
gamma2 = @. 0.5 * (dxy + dyx)

# Get magnification
mu = @. 1.0 / ((1.0 - kappa)^2 - (gamma1^2 + gamma2^2))
fig, ax = Lenses.plot_sky(x, y)
heatmap!(ax, x[:,1], y[1,:], abs.(mu); colormap=:viridis, colorrange=(1, 100), colorscale=log10)
display(fig)
